# SPARCS Asthma Hospitalization Extractor and Aggregator

**Purpose:** Read SPARCS annual CSV files, normalize ZIP3, diagnosis, payer, and numeric fields, filter for asthma hospitalizations, compute derived indicators, and aggregate results for ZIP3/age-group and borough-level analysis.

## Data sources

All raw data is sourced from the **New York State Department of Health's Statewide Planning and Research Cooperative System (SPARCS)**, which collects inpatient discharge records from all hospitals in New York State.

- **Portal:** [health.data.ny.gov](https://health.data.ny.gov) → "Hospital Inpatient Discharges (SPARCS De-Identified)"
- **Years covered:** 2010–2024 (15 annual files, ~10.3 GB total)
- **Local storage:** `data_csv/SPARCS/` as `Hospital_Inpatient_Discharges_SPARCS_{yyyy}.csv`

### Yearly dataset IDs and download links

| Year | Dataset ID | Records | CCSR/CCS |
|------|-----------|---------|----------|
| 2010–2018 | See portal | ~2–2.6M each | CCS |
| 2019–2024 | See portal | ~870K–2.3M each | **CCSR** (2019+) |

Full dataset IDs and direct links available on [health.data.ny.gov](https://health.data.ny.gov).

### Key columns (across all years)

Facility info: Health Service Area, Hospital County, Facility ID/Name  
Patient demographics: Age Group, Gender, Race, Ethnicity, Zip Code  
Admission/stay: Length of Stay, Type of Admission, Patient Disposition, Discharge Year  
Diagnoses & procedures: CCS/CCSR Code & Description  
Severity: APR Severity of Illness, APR Risk of Mortality, APR DRG Code  
Financial: Total Charges, Total Costs, Source of Pay/Payment Typology  
Other: Emergency Department Indicator (2019+), Birth Weight (2019+)

### Schema change note

- **2010–2018:** CCS (Clinical Classifications Software) — `CCS Diagnosis Code`, `CCS Diagnosis Description`
- **2019–2024:** CCSR (Refined) — `CCSR Diagnosis Code`, `CCSR Diagnosis Description`

### Download method

CSVs downloaded via Socrata CSV export API:

```
curl -L -o "Hospital_Inpatient_Discharges_SPARCS_{yyyy}.csv" \
  "https://health.data.ny.gov/api/views/{dataset-id}/rows.csv?accessType=DOWNLOAD"
```

**Primary outputs:**
- `data_csv/SPARCS_asthma_hospitalizations_by_zip3_2010-2024.csv`
- `data_csv/SPARCS_by_borough_2010-2024.csv`

**Dependencies:**
- Python 3.8+
- `pandas`, `numpy`

**How to run:**
1. Activate the project virtual environment if present:
```bash
source .venv/bin/activate
```
2. Place SPARCS annual CSV files in `data_csv/SPARCS/`.
3. Run this notebook in Jupyter or execute the code cells to extract, normalize, filter, and aggregate asthma hospitalizations.

**Notes & reproducibility:**
- The notebook processes large CSV files in chunks, standardizes variable names across year-specific schemas, and maps diagnosis descriptions to asthma cases.
- It also computes derived indicators like ED visit counts, Medicaid case flags, and average costs before aggregating by ZIP3/age group and borough.

**Author / Maintainer:** Mayank Anand — mayank.anand3007@gmail.com

In [1]:
import os
import re
import glob
import pandas as pd
import numpy as np

In [2]:
sparcs_path = "data_csv/SPARCS/"
sparcs_files = glob.glob(os.path.join(sparcs_path, "*.csv"))
asthma_list = []

for file in sparcs_files:

    filename = os.path.basename(file)

    # Extract year from filename, e.g. ..._2019.csv
    match = re.search(r'_(\d{4})(?=\.csv$|_)', filename)

    if match:
        year = int(match.group(1))
    else:
        raise ValueError(f"Could not extract year from filename: {filename}")

    chunk_iter = pd.read_csv(file, chunksize=500000, low_memory=False)
    
    for chunk in chunk_iter:
        
        # -------------------------
        # Standardize ZIP column
        # -------------------------
        if "Zip Code - 3 digits" in chunk.columns:
            chunk = chunk.rename(columns={"Zip Code - 3 digits": "zip3"})
        elif "Zip Code" in chunk.columns:
            chunk = chunk.rename(columns={"Zip Code": "zip3"})
        else:
            raise KeyError("No ZIP column found in this file.")
        
        # Clean ZIP3
        chunk["zip3"] = chunk["zip3"].astype(str).str.extract(r'(\d{3})')
        chunk = chunk.dropna(subset=["zip3"])
        
        # Ensure Age Group exists
        if "Age Group" not in chunk.columns:
            raise KeyError("Age Group column not found.")
        
        # -------------------------
        # Standardize diagnosis description column (CCSR vs CCS)
        # -------------------------
        if "CCSR Diagnosis Description" in chunk.columns:
            diag_col = "CCSR Diagnosis Description"
        elif "CCS Diagnosis Description" in chunk.columns:
            diag_col = "CCS Diagnosis Description"
        else:
            raise KeyError(f"No diagnosis description column found in: {filename}")
        
        # -------------------------
        # Filter asthma cases
        # -------------------------
        asthma = chunk[
            chunk[diag_col].str.contains("Asthma", case=False, na=False)
        ].copy()
        
        # Normalize to a single column name for downstream use
        if diag_col != "CCSR Diagnosis Description":
            asthma = asthma.rename(columns={diag_col: "CCSR Diagnosis Description"})
        
        if asthma.empty:
            continue
        
        # -------------------------
        # Clean currency fields (strip $ and commas before converting)
        # -------------------------
        for currency_col in ["Total Charges", "Total Costs"]:
            if currency_col in asthma.columns:
                asthma[currency_col] = (
                    asthma[currency_col]
                    .astype(str)
                    .str.replace(r'[\$,\s]', '', regex=True)
                    .replace('', np.nan)
                    .replace('nan', np.nan)
                    .pipe(pd.to_numeric, errors='coerce')
                )

        # -------------------------
        # Map text-based Risk of Mortality to numeric
        # -------------------------
        mortality_map = {
            "Minor":    1,
            "Moderate": 2,
            "Major":    3,
            "Extreme":  4
        }
        if "APR Risk of Mortality" in asthma.columns:
            asthma["APR Risk of Mortality"] = (
                asthma["APR Risk of Mortality"]
                .astype(str)
                .str.strip()
                .map(mortality_map)
            )

        # -------------------------
        # Clean remaining numeric fields
        # -------------------------
        numeric_cols = [
            "Length of Stay",
            "APR Severity of Illness Code",
            "APR Risk of Mortality",
        ]
        for col in numeric_cols:
            if col in asthma.columns:
                asthma[col] = pd.to_numeric(asthma[col], errors="coerce")
        
        # -------------------------
        # Derived indicators
        # -------------------------
        asthma["ED_Visit"] = np.where(
            asthma["Emergency Department Indicator"].astype(str).str.upper() == "Y", 1, 0
        )
        
        if "Payment Typology 1" in asthma.columns:
            asthma["Medicaid"] = np.where(
                asthma["Payment Typology 1"].str.contains("Medicaid", case=False, na=False),
                1, 0
            )
        else:
            asthma["Medicaid"] = 0
        
        # -------------------------
        # Group by ZIP3 + Age Group
        # -------------------------
        grouped = asthma.groupby(["zip3", "Age Group"]).agg(
            Asthma_Cases=("zip3", "size"),
            ED_Visits=("ED_Visit", "sum"),
            Avg_Length_of_Stay=("Length of Stay", "mean"),
            Avg_Severity=("APR Severity of Illness Code", "mean"),
            Avg_Mortality_Risk=("APR Risk of Mortality", "mean"),
            Total_Charges=("Total Charges", "sum"),
            Total_Costs=("Total Costs", "sum"),
            Avg_Costs=("Total Costs", "mean"),
            Medicaid_Cases=("Medicaid", "sum")
        ).reset_index()
        
        grouped["Year"] = year
        asthma_list.append(grouped)

# ------------------------------------
# Combine all chunks
# ------------------------------------
if not asthma_list:
    print("No asthma cases found across all files.")
else:
    sparcs_combined = pd.concat(asthma_list, ignore_index=True)

    sparcs_final = sparcs_combined.groupby(
        ["Year", "zip3", "Age Group"]
    ).agg({
        "Asthma_Cases":       "sum",
        "ED_Visits":          "sum",
        "Avg_Length_of_Stay": "mean",
        "Avg_Severity":       "mean",
        "Avg_Mortality_Risk": "mean",
        "Total_Charges":      "sum",
        "Total_Costs":        "sum",
        "Avg_Costs":          "mean",
        "Medicaid_Cases":     "sum"
    }).reset_index()

    sparcs_final["ED_Rate"]       = sparcs_final["ED_Visits"]      / sparcs_final["Asthma_Cases"]
    sparcs_final["Medicaid_Rate"] = sparcs_final["Medicaid_Cases"] / sparcs_final["Asthma_Cases"]

    print(sparcs_final.head())

   Year zip3    Age Group  Asthma_Cases  ED_Visits  Avg_Length_of_Stay  \
0  2010  100      0 to 17           907        837            2.245865   
1  2010  100     18 to 29           233        208            2.506438   
2  2010  100     30 to 49           678        583            3.069322   
3  2010  100     50 to 69          1206       1056            4.000830   
4  2010  100  70 or Older           691        560            4.615051   

   Avg_Severity  Avg_Mortality_Risk  Total_Charges  Total_Costs    Avg_Costs  \
0      1.304300            1.028666    10984565.92   6114083.31  6740.995932   
1      1.442060            1.107296     3667256.97   1617234.12  6940.918970   
2      1.849558            1.225664    11509724.53   5086873.47  7502.763230   
3      2.029022            1.450249    29351122.32  11581206.97  9602.990854   
4      2.175109            2.046310    18719426.12   6721517.28  9727.231954   

   Medicaid_Cases   ED_Rate  Medicaid_Rate  
0               0  0.922822  

In [3]:
# sparcs_final.to_csv("data_csv/SPARCS_asthma_hospitalizations_by_zip3_2010-2024.csv", index=False)

In [4]:
df = sparcs_final.copy()
zip3_to_borough = {
    # NYC - Manhattan (New York County)
    100: "Manhattan",
    101: "Manhattan",

    # NYC - Bronx (Bronx County)
    104: "Bronx",

    # NYC - Staten Island (Richmond County)
    103: "Staten Island",

    # NYC - Queens (Queens County)
    110: "Queens",
    111: "Queens",
    113: "Queens",
    114: "Queens",
    116: "Queens",

    # NYC - Brooklyn (Kings County)
    112: "Brooklyn",

    # NYC - Other / Mixed NYC (used by USPS for PO Boxes, unique zips, etc.)
    102: "Manhattan",   # PO Boxes / unique Manhattan zips
    105: "Bronx",       # Bronx overlap
    106: "Bronx",       # Bronx overlap

    # New York State - Outside NYC
    107: "Westchester County",
    108: "Westchester County",
    109: "Westchester County",
    115: "Nassau County",
    117: "Nassau County",
    118: "Suffolk County",
    119: "Suffolk County",

    # Hudson Valley / Capital Region
    120: "Albany Area",
    121: "Albany Area",
    122: "Albany Area",
    123: "Schenectady/Albany Area",
    124: "Hudson Valley",
    125: "Hudson Valley",
    126: "Hudson Valley",
    127: "Hudson Valley",
    128: "Catskills",
    129: "Hudson Valley",

    # Central/Western NY
    130: "Syracuse Area",
    131: "Syracuse Area",
    132: "Syracuse Area",
    133: "Utica Area",
    134: "Utica Area",
    135: "Utica Area",
    136: "Watertown Area",
    137: "Binghamton Area",
    138: "Binghamton Area",
    139: "Binghamton Area",

    # Western NY
    140: "Buffalo Area",
    141: "Buffalo Area",
    142: "Buffalo Area",
    143: "Niagara Falls Area",
    144: "Rochester Area",
    145: "Rochester Area",
    146: "Rochester Area",
    147: "Corning/Elmira Area",
    148: "Ithaca/Elmira Area",
    149: "Elmira Area",
}

# filter to keep only NYC's five boroughs
nyc_zip3s = {100, 101, 102, 103, 104, 105, 106, 110, 111, 112, 113, 114, 116}

df = df[df['zip3'].isin(nyc_zip3s)]

nyc_boroughs = {"Manhattan", "Bronx", "Brooklyn", "Queens", "Staten Island"}

# Map borough
df['Borough'] = df['zip3'].map(zip3_to_borough)
df = df[df['Borough'].isin(nyc_boroughs)]

# Check if any zip3 wasn't mapped
unmapped = df[df['Borough'].isna()]['zip3'].unique()
if len(unmapped) > 0:
    print(f"Unmapped zip3s: {unmapped}")

# Aggregate by Year and Borough
agg_funcs = {
    'Asthma_Cases': 'sum',
    'ED_Visits': 'sum',
    'Avg_Length_of_Stay': 'mean',
    'Avg_Severity': 'mean',
    'Avg_Mortality_Risk': 'mean',
    'Total_Charges': 'sum',
    'Total_Costs': 'sum',
    'Avg_Costs': 'mean',
    'Medicaid_Cases': 'sum',
    'ED_Rate': 'mean',
    'Medicaid_Rate': 'mean'
}

df_agg = df.groupby(['Year', 'Borough']).agg(agg_funcs).reset_index()

# Optional: round numeric columns for readability
numeric_cols = df_agg.select_dtypes('number').columns
df_agg[numeric_cols] = df_agg[numeric_cols].round(2)

df_agg.to_csv("data_csv/SPARCS_by_borough_2010-2024.csv", index=False)